# Trust Game: two-run fixed-effects FEAT

**Author:** Smith Lab  
**Updated:** 2026-08-18  
**License:** MIT

This notebook combines the two model-1 activation outputs from notebook 02. It is within-participant fixed effects, not a group analysis.

## What you will learn

1. Verify both run-level inputs.
2. Render and inspect the production L1→L2 path contract.
3. Run fixed effects and inspect reciprocation > defection.


## 1. Load pinned FSL and locate outputs

In [ ]:
import module
await module.load('fsl/6.0.7.22')
await module.list()

In [ ]:
%pip install -q nibabel matplotlib watermark
from pathlib import Path
import os, subprocess
import nibabel as nib
import matplotlib.pyplot as plt
from IPython.display import IFrame, Image, display

In [ ]:
SUBJECT='10317'; SESSION='01'; WORKSPACE=Path.home()/'trust_teaching'; FSL_DIR=WORKSPACE/'fsl'
REPO=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'code'/'L2stats.sh').is_file()),None)
if REPO is None: raise FileNotFoundError('Run inside a clone of rf1-sra-trust.')
env={**os.environ,'FSL_DERIVATIVES_ROOT':str(FSL_DIR)}
inputs=[]
for run in ('1','2'):
    feat=FSL_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/f'L1_task-trust_ses-{SESSION}_model-1_type-act_run-{run}_sm-5.feat'
    if not (feat/'stats'/'cope18.nii.gz').is_file() or not (feat/'cluster_mask_zstat1.nii.gz').is_file(): raise FileNotFoundError(f'Complete notebook-02 output required: {feat}')
    inputs.append(feat)
print('\n'.join(map(str,inputs)))

## 2. Render and verify fixed effects

`L2stats.sh` defaults `FSLSUB_PARALLEL=1`, preventing an uncontrolled inner worker pool while batch `--jobs` remains the outer control.

In [ ]:
subprocess.run(['bash',str(REPO/'code'/'L2stats.sh'),SUBJECT,'act','--session',SESSION,'--render-only'],env=env,check=True)
rendered=FSL_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/f'L2_sub-{SUBJECT}_task-trust_ses-{SESSION}_model-1_type-act.fsf'; text=rendered.read_text()
for required in (*map(str,inputs),'set fmri(mixed_yn) 3','set fmri(ncopeinputs) 18'):
    if required not in text: raise AssertionError(f'Missing L2 contract: {required}')
print(rendered)
print('\n'.join(line for line in text.splitlines() if any(x in line for x in ('outputdir','feat_files','mixed_yn','ncopeinputs'))))

## 3. Run L2 fixed effects

In [ ]:
OVERWRITE_INCOMPLETE=False
command=['bash',str(REPO/'code'/'L2stats.sh'),SUBJECT,'act','--session',SESSION]
if OVERWRITE_INCOMPLETE: command.append('--overwrite')
subprocess.run(command,env=env,check=True)

## 4. Inspect the combined result

Contrast 10 remains the established reciprocation > defection contrast. This estimates the participant effect shared across runs 1 and 2.

In [ ]:
gfeat=FSL_DIR/f'sub-{SUBJECT}'/f'ses-{SESSION}'/f'L2_task-trust_ses-{SESSION}_model-1_type-act_sm-5.gfeat'; cope=gfeat/'cope10.feat'
if not (cope/'cluster_mask_zstat1.nii.gz').is_file(): raise FileNotFoundError(gfeat)
display(Image(filename=str(cope/'design.png'))); display(IFrame(src=str(cope/'report.html'),width='100%',height=650))
img=nib.load(cope/'stats'/'zstat1.nii.gz'); data=img.get_fdata(); z=data.shape[2]//2
plt.figure(figsize=(6,5)); plt.imshow(data[:,:,z].T,cmap='coolwarm',origin='lower',vmin=-5,vmax=5); plt.title('L2 fixed effects: rec > defect'); plt.axis('off'); plt.colorbar(label='Z'); plt.show()

## 5. Interpretation and dependencies

This is within-participant fixed effects, not population inference. Preserve both FEAT reports and the repository commit when recording a run.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions
await module.list()